# 목표

이 노트북은 Ray Train의 분산 XGBoost(XGBoostTrainer)를 활용하여 **MNIST 데이터셋**을 다중 클래스 분류하는 방법을 실습하기 위한 파일입니다. 아래 각 셀의 지시사항에 따라 빈 셀(Code)을 추가하여 코드를 직접 작성하세요.

**[Step 1: 라이브러리 설치 및 환경 설정]**

- 환경 변수 `RAY_TRAIN_V2_ENABLED`를 `"1"`로 설정하세요.
- 필요한 패키지를 설치하는 명령어를 작성하세요:
  설치할 패키지: `ray[train]`, `xgboost`, `xgboost_ray`, `torchvision`, `pandas`, `matplotlib`, `ipywidgets`

**[Step 2: 모듈 임포트 및 Ray 초기화]**

- 데이터 처리 및 시각화: `tqdm.auto.tqdm`, `pandas`, `numpy`, `matplotlib.pyplot`, `torchvision`
- Ray 관련 모듈: `ray`, `ray.train` (ScalingConfig, RunConfig 등)
- Ray Train XGBoost 모듈: `XGBoostTrainer`, `XGBoostPredictor`, `BatchPredictor` (ray.train.batch_predictor)
- `ray.init()`를 명시적으로 호출하여 임시 Ray 클러스터를 생성하여 활용하세요.

**[Step 3: MNIST 데이터셋 로드 및 Ray Dataset 변환]**

- `torchvision.datasets.MNIST`를 사용하여 학습(train) 및 검증(test) 데이터를 다운로드하세요.
- 불러온 28x28 형태의 이미지 데이터를 784차원의 1D 배열(Flatten)로 변환하세요.
- 픽셀 데이터(pixel_0 ~ pixel_783)와 레이블(label) 값을 포함한 Pandas DataFrame으로 변환하세요.
- `ray.data.from_pandas()`를 호출하여 Pandas DataFrame을 Ray Dataset 객체로 만들어 반환(train_ds, valid_ds)하는 함수를 작성하고 실행하세요.

**[Step 4: 분산 XGBoost 학습 실행]**

- `XGBoostTrainer` 객체를 생성하세요.
- 워커 수(`num_workers`)를 2개 이상 사용하는 `ScalingConfig`를 지정하세요.
- 라벨 컬럼을 `"label"`로, `num_boost_round`를 적절히(예: 20) 설정하세요.
- 학습 파라미터(`params`) 딕셔너리에 다중 분류 목적(`"objective": "multi:softmax"`), 분류할 클래스 수(`"num_class": 10`), 그리고 평가 지표(`"eval_metric": ["mlogloss", "merror"]`) 및 트리 메소드(`"tree_method": "hist"`)를 입력하세요.
- 학습 및 검증 데이터를 딕셔너리로 넘겨준 뒤 `trainer.fit()`을 호출해 학습을 수행하세요.

**[Step 5: 학습 결과 및 측정 지표 확인]**

- 학습 결과 객체(`result`)에서 `metrics_dataframe`을 가져오세요.
- `matplotlib.pyplot`을 사용하여 에폭(Boosting Round)에 따른 학습/검증의 **Log Loss(`mlogloss`)** 곡선과 **분류 오차(`merror`)** 곡선을 나란히(서브플롯 등) 시각화하세요.

**[Step 6: Batch Inference (분산 추론) 실행 및 결과 시각화]**

- 평가를 위해 검증 데이터셋(`valid_ds`)에서 `drop_columns`을 사용해 `"label"` 컬럼을 제거하세요.
- 최상의 체크포인트(`result.checkpoint`)와 `XGBoostPredictor` 클래스를 사용해 `BatchPredictor`를 초기화하세요.
- `predictor.predict()`를 호출해 Ray 클러스터 기반 분산 추론(예측)을 수행하고, 결과를 Pandas DataFrame으로 변환하세요.
- 원래의 검증 데이터(`valid_ds`)와 예측 결과를 비교하기 위해, 랜덤하게 5개의 데이터를 뽑아 원본 이미지(28x28 재구성)와 예측값(Pred), 실제 레이블(Actual)을 겹쳐서 출력(matplotlib)하세요.